#### setup

In [ ]:
# make library code importable
import sys
from pathlib import Path
ROOT = Path.cwd().parents[3]
TREE_DIR = ROOT / "experiments" / "supplementary" / "baselines" / "tree_ranker"
sys.path.append(str(ROOT / "library"))
sys.path.append(TREE_DIR)

# generic imports
import os
import json
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader

# library imports
from data_utils import *
from models import *
from training import *
from eval_utils import *

In [ ]:
# install dependencies - only once
%pip install causalml
%pip install -U "statsmodels>=0.14.5"

In [ ]:
# custom library imports - restart kernel after installation if needed
from tree_library import *

In [ ]:
# set device and seed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#### data

In [ ]:
# set dataset parameters
dataset = 'synthetic'
train_size = 1000

In [ ]:
# set confounders
confounders = ['x0', 'x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8', 'x9']
input_dim = len(confounders)

In [ ]:
# read
data_path = ROOT / "data" / "datasets" / f"{dataset}.csv"
df = pd.read_csv(data_path, index_col=0)

#### helpers

In [ ]:
def load_nuisance_models(ns_seed_dir, input_dim, device):
    """Helper to load nuisance models. Set the hidden dim correctly!"""
    paths = {
        "e": ns_seed_dir / "prop_model.pt",
        "m0": ns_seed_dir / "mu0_model.pt",
        "m1": ns_seed_dir / "mu1_model.pt"}

    for name, path in paths.items():
        if not path.exists():
            raise FileNotFoundError(f"Missing nuisance checkpoint {name}: {path}")

    prop_model = ClassificationHead(input_dim=input_dim, hidden_dim=128).to(device)
    prop_model.load_state_dict(torch.load(paths["e"], map_location=device, weights_only=True))

    m0_model = RegressionHead(input_dim=input_dim, hidden_dim=64).to(device)
    m0_model.load_state_dict(torch.load(paths["m0"], map_location=device, weights_only=True))

    m1_model = RegressionHead(input_dim=input_dim, hidden_dim=64).to(device)
    m1_model.load_state_dict(torch.load(paths["m1"], map_location=device, weights_only=True))

    return prop_model, m0_model, m1_model

In [ ]:
def init_tree_ranker(input_dim, params, seed):
    return RandomForestAUTOC(
        n_estimators=params['n_estimators'],
        max_depth=params['max_depth'],
        max_samples=params['max_samples'],
        min_samples_split=params['min_samples_split'],
        min_samples_leaf=params['min_samples_leaf'],
        min_impurity=params['min_impurity'],
        method=1,              
        n_jobs=0,
        seed=seed)

In [ ]:
def evaluate_tree_ranker(model, test_df, confounders):
    eval_df = test_df.copy()

    # predict
    X_test = eval_df[confounders].to_numpy(dtype=float)
    eval_df["score"] = model.predict(X_test)

    # evaluate
    ranked = eval_df.sort_values("score", ascending=False).copy()

    return {
        "model": "TreeRanker",
        "autoc": float(autoc(ranked)),
        "policy_value": float(policy_value(ranked))}

#### train and evaluate

In [ ]:
# load configs
configs = pd.read_csv(f'./configs/tree_ranker.csv', index_col=0)

# set configs
row = configs.iloc[0]
params = dict(
    max_depth=int(row['max_depth']),
    min_samples_leaf=int(row['min_samples_leaf']),
    min_samples_split=int(row['min_samples_split']),
    n_estimators=int(row['n_estimators']),
    max_samples=float(row['max_samples']),
    min_impurity=-1e-6)

In [ ]:
# set directories
ns_dir = ROOT / "experiments" / "nuisances" / "chkpts" / dataset

In [ ]:
# init collector
all_results = []

# loop over seeds
for seed in range(5):

    # track progress
    print(f" -> Seed {seed}, size {train_size}")
    set_seed(seed)

    # get training and testing data
    _, _, train_df, val_df, test_df = make_splits(df=df, train_size=train_size, seed=seed)

    # load nuisance models
    ns_seed_dir = ns_dir / f"size_{train_size}" / f"seed_{seed}"
    prop_model, m0_model, m1_model = load_nuisance_models(ns_seed_dir=ns_seed_dir, input_dim=input_dim, device=device)

    # add pseudo outcomes to dataframes
    train_df = compute_dr_scores(train_df, confounders, prop_model, m0_model, m1_model, device)

    # store data as array
    X = train_df[confounders].to_numpy(dtype=float)
    y_dr = train_df['DR'].to_numpy(dtype=float)

    # init and train model
    model = init_tree_ranker(input_dim, params, seed)
    model.fit(X, y_dr)
    
    # evaluate and store
    row = evaluate_tree_ranker(model=model, test_df=test_df, confounders=confounders)
    row.update({"seed": seed, "train_size": train_size})
    all_results.append(row)

# summarize
df_all = pd.DataFrame(all_results)
summary = (df_all.groupby(["train_size", "model"])[["autoc", "policy_value"]].agg(["mean", "std"]).reset_index())